In [1]:
import xarray as xr
import pandas as pd
import numpy as np
from scipy.stats import linregress
#from scipy.signal import detrend
from sklearn.linear_model import LinearRegression

import matplotlib.pyplot as plt

In [2]:
def detrend_with_trend(x, y):
    """Return both detrended series and fitted trend (as a Series)."""
    mask = np.isfinite(x) & np.isfinite(y)
    x_valid = x[mask].values.reshape(-1, 1)
    y_valid = y[mask].values

    model = LinearRegression().fit(x_valid, y_valid)
    trend = model.predict(x.values.reshape(-1, 1))
    detrended = y - trend
    
    return detrended, trend



In [3]:
def classify_precip(row):
    code = row['WEATHER']
    date = row['DATE']
    year = date.year  # extract year from datetime

    if code == 5:
        return 'Non-Precipitation'
    elif code in [6, 7]:
        return 'Non-Precipitation' if year >= 2007 else 'Precipitation'
    else:
        return 'Precipitation'

In [4]:
state_dictAB = {
    1: 'AL', 2: 'AK', 4: 'AZ', 5: 'AR', 6: 'CA', 8: 'CO', 9: 'CT', 10: 'DE',
    11: 'DC', 12: 'FL', 13: 'GA', 15: 'HI', 16: 'ID', 17: 'IL', 18: 'IN',
    19: 'IA', 20: 'KS', 21: 'KY', 22: 'LA', 23: 'ME', 24: 'MD', 25: 'MA',
    26: 'MI', 27: 'MN', 28: 'MS', 29: 'MO', 30: 'MT', 31: 'NE', 32: 'NV',
    33: 'NH', 34: 'NJ', 35: 'NM', 36: 'NY', 37: 'NC', 38: 'ND', 39: 'OH',
    40: 'OK', 41: 'OR', 42: 'PA', 44: 'RI', 45: 'SC', 46: 'SD', 47: 'TN',
    48: 'TX', 49: 'UT', 50: 'VT', 51: 'VA', 53: 'WA', 54: 'WV', 55: 'WI',
    56: 'WY'
}

In [5]:
excluded_states = {'DC', 'AK', 'HI', 'PR', 'PRCP'}
months_list=[12,1,2]

In [6]:
cluster_names = {
    0: "Alaskan Ridge/Pacific Ridge",
    1: "Greenland High",
    2: "Pacific Trough",
    3: "West Coast High"
}

In [7]:
# Contains FARS data for weather-related crashes (Downloaded by hand from FARS database)
fars_file='../data/FARS/FARSNO2UPDATED_FIXED.csv'

# Maps precip anomalies by state and day (created by MakePrecipSWEStates_daily.py)
precip_file='../data/precip/state_precip_chirps.csv'

# Composite State precip anomalies by weather regime  (created by MakePrecipSWEStates_wxregimes.py)
wx_regimes_file='../data/precip/state_precip_wxregimes.csv'

# Maps a Month to Nino34 SST anomaly (downloaded from NOAA/PSL)
enso_file='../data/ENSO/ENSOMONTHLY.csv'

# Maps a day to a weather regime (created by Kathy using weather regimes code)
cluster_file='../data/wxregimes/kmeans_4cluster_DJF_1981-2019_NA.nc'

In [8]:
start_year=1981
end_year=2023

# Read and Clean FARS Data

In [9]:
# FARS DATA

df_fars=pd.read_csv(fars_file)
print("Number of FWRCs :",len(df_fars))
print(df_fars)

# map states
df_fars['STATE_ABBR'] = df_fars['STATE'].map(state_dictAB)

# remove excluded states
df_fars=df_fars[~df_fars["STATE_ABBR"].isin(excluded_states)]
print("Number of FWRCs CONUS:",len(df_fars))

# Subset Years in fars
df_fars = df_fars[(df_fars["YEAR"] >= start_year) & (df_fars["YEAR"] <= end_year)]
print("Number of FWRCs CONUS 1981-2023:",len(df_fars))

# Build datetime column in df_fars
df_fars["DATE"] = pd.to_datetime( df_fars["YEAR"].astype(str).str.zfill(4) + 
                                  df_fars["MONTH"].astype(str).str.zfill(2) + 
                                  df_fars["DAY"].astype(str).str.zfill(2), format="%Y%m%d", 
                                  errors="coerce")
df_fars["DATE"] = pd.to_datetime(df_fars["DATE"])
                                 
# Identify & Keep Precip-only weather
df_fars["precip_class"] = df_fars.apply(classify_precip, axis=1)
df_fars_clean = df_fars[df_fars["precip_class"] == "Precipitation"].copy()
df_fars_clean.drop(columns=["precip_class"], inplace=True)
print("Number of FWRCs Precip-only 1981-2023:",len(df_fars_clean))
print('% Precip FWRCs: ',len(df_fars_clean)/len(df_fars)*100)

# Drop other unnecessary columns
df_fars_clean.drop(columns=["STATE","WEATHER", "MONTH","DAY","YEAR","HOUR","MINUTE", "DAY_WEEK","COUNTY","CITY"], inplace=True)

# Re-order columns
cols = ["STATE_ABBR", "DATE"] + [c for c in df_fars_clean.columns if c not in ["STATE_ABBR", "DATE"]]
df_fars_clean = df_fars_clean[cols]

# Rename for clarity
df_fars_clean = df_fars_clean.rename(columns={"FATALS": "FATAL_CRASH_COUNT"}).reset_index(drop=True)

Number of FWRCs : 207798
        STATE  COUNTY  CITY  MONTH  DAY  YEAR  HOUR  MINUTE  WEATHER  FATALS  \
0           1      89  2225      1    8  1975    13      20        2       1   
1           1      69  2961      1    4  1975     1      15        2       1   
2           1     101  2130      1    3  1975    17      44        2       1   
3           1     117     0      1   10  1975    15      20        2       1   
4           1      83   120      1    8  1975    14      37        2       1   
...       ...     ...   ...    ...  ...   ...   ...     ...      ...     ...   
207793      9       9   669      2   18  2021    17      52        4       1   
207794      9       9   790      2   19  2021    23      34        4       1   
207795      9       9   810      1   13  2020    16      22        2       1   
207796      9       9   810      7    9  2021    21      48        2       1   
207797      9       9   854      5   16  2020     1       1        2       1   

        DAY_WE

In [10]:
df_djf = df_fars_clean[df_fars_clean['DATE'].dt.month.isin(months_list)]
print("Number of FWRCs Precip-only DJF 1981-2023:",len(df_djf))

Number of FWRCs Precip-only DJF 1981-2023: 53613


In [11]:
df_fars_clean

,STATE_ABBR,DATE,FATAL_CRASH_COUNT
0,AL,1981-01-06,1
1,AL,1981-01-21,1
2,AL,1981-01-20,1
3,AL,1981-01-20,1
4,AL,1981-02-01,1
...,...,...,...
155847,CT,2021-02-18,1
155848,CT,2021-02-19,1
155849,CT,2020-01-13,1
155850,CT,2021-07-09,1


In [12]:
monthly_counts = (
    df_fars_clean
    .groupby([df_fars_clean['DATE'].dt.year, 
              df_fars_clean['DATE'].dt.month])
    .size()
    .unstack(fill_value=0)
)
print(monthly_counts) 

DATE    1.0   2.0   3.0   4.0   5.0   6.0   7.0   8.0   9.0   10.0  11.0  12.0
DATE                                                                          
1981.0   289   429   327   312   355   310   307   302   296   471   395   611
1982.0   516   397   414   334   296   340   217   224   306   356   609   726
1983.0   452   397   559   510   355   258   147   191   270   410   612   735
1984.0   378   396   419   397   333   206   284   236   283   452   478   597
1985.0   539   399   371   248   266   316   284   283   262   457   804   467
1986.0   379   433   319   282   283   274   265   286   306   442   618   599
1987.0   481   395   375   302   281   292   216   310   367   313   559   663
1988.0   440   405   375   367   269   153   262   236   308   361   560   498
1989.0   428   508   483   307   334   387   294   247   413   406   392   535
1990.0   502   442   446   323   392   208   273   293   216   321   360   621
1991.0   469   338   480   366   300   174   178   2

# 2 Create Daily database

In [13]:
# Get the State Precip data
df_state_precip=pd.read_csv(precip_file)
print(df_state_precip)

# remove excluded states
df_state_precip=df_state_precip[~df_state_precip["STATE_ABBR"].isin(excluded_states)]

# Keep only desired columns and rename for clarity
df_state_precip = df_state_precip[["DATE", "STATE_ABBR", "ANOM_MEAN"]].rename(
    columns={"ANOM_MEAN": "PRECIP_ANOM"}
)

# Make datetime64 & subset years
df_state_precip["DATE"] = pd.to_datetime(df_state_precip["DATE"])
df_state_precip = df_state_precip[(df_state_precip["DATE"].dt.year >= start_year) & (df_state_precip["DATE"].dt.year <= end_year)]

# Get Daily Crashes
df_crashes_daily = (
    df_fars_clean
    .groupby(['STATE_ABBR', 'DATE'])['FATAL_CRASH_COUNT']
    .sum()
    .reset_index()
)

df_daily_state = df_state_precip.merge(
    df_crashes_daily,
    on=['STATE_ABBR', 'DATE'],
    how='left'
)
df_daily_state['FATAL_CRASH_COUNT'] = df_daily_state['FATAL_CRASH_COUNT'].fillna(0)

df_daily_state

              DATE  ANOM_MEAN    ANOM_SUM  STATE  STATEFP STATE_ABBR
0       1981-01-01  -4.391150 -917.750447      1        1         AL
1       1981-01-02  -4.417280 -923.211495      1        1         AL
2       1981-01-03  -4.426943 -925.231093      1        1         AL
3       1981-01-04  -4.419716 -923.720710      1        1         AL
4       1981-01-05  -4.411256 -921.952587      1        1         AL
...            ...        ...         ...    ...      ...        ...
785245  2023-12-27  -2.748115  -41.221719     72       72         PR
785246  2023-12-28  -3.143802  -47.157032     72       72         PR
785247  2023-12-29  -3.105917  -46.588751     72       72         PR
785248  2023-12-30  -3.064111  -45.961672     72       72         PR
785249  2023-12-31  13.891354  208.370308     72       72         PR

[785250 rows x 6 columns]


,DATE,STATE_ABBR,PRECIP_ANOM,FATAL_CRASH_COUNT
0,1981-01-01,AL,-4.391150,0.0
1,1981-01-02,AL,-4.417280,0.0
2,1981-01-03,AL,-4.426943,0.0
3,1981-01-04,AL,-4.419716,0.0
4,1981-01-05,AL,-4.411256,0.0
...,...,...,...,...
753835,2023-12-27,WY,-0.512046,0.0
753836,2023-12-28,WY,-0.460988,0.0
753837,2023-12-29,WY,-0.530808,0.0
753838,2023-12-30,WY,-0.528172,0.0


# 3 Add Nino3.4 Data

In [14]:
df_enso = pd.read_csv(enso_file).replace(-99.99, np.nan)

# Create a monthly timestamp in df_enso
df_enso["DATE"] = pd.to_datetime(
    df_enso["YEAR"].astype(str).str.zfill(4) +
    df_enso["MONTH"].astype(str).str.zfill(2),
    format="%Y%m"
)

# Keep only needed columns and rename for clarity
df_enso = df_enso[["DATE", "SST_VALUE"]].rename(columns={"SST_VALUE": "NINO34_SST"})

# Extract monthly period from daily crash dataframe
df_daily_state["MONTH_START"] = df_daily_state["DATE"].dt.to_period("M").dt.to_timestamp()

# Merge using the monthly timestamp
df_merged = df_daily_state.merge( df_enso, left_on="MONTH_START", right_on="DATE", how="left" )

# Clean up join artifacts
df_merged = df_merged.drop(columns=["DATE_y","MONTH_START"]).rename(columns={"DATE_x": "DATE"})

# Ensure datetime64 for DATE is retained
df_merged["DATE"] = pd.to_datetime(df_merged["DATE"])

df_merged

,DATE,STATE_ABBR,PRECIP_ANOM,FATAL_CRASH_COUNT,NINO34_SST
0,1981-01-01,AL,-4.391150,0.0,-0.36
1,1981-01-02,AL,-4.417280,0.0,-0.36
2,1981-01-03,AL,-4.426943,0.0,-0.36
3,1981-01-04,AL,-4.419716,0.0,-0.36
4,1981-01-05,AL,-4.411256,0.0,-0.36
...,...,...,...,...,...
753835,2023-12-27,WY,-0.512046,0.0,2.03
753836,2023-12-28,WY,-0.460988,0.0,2.03
753837,2023-12-29,WY,-0.530808,0.0,2.03
753838,2023-12-30,WY,-0.528172,0.0,2.03


# 4 Calculate and Add ENSO Precip Anomalies

In [15]:
def add_nino34_pred(group):
    # Drop missing values
    group_valid = group.dropna(subset=["PRECIP_ANOM", "NINO34_SST"])
    
    # Skip if not enough data or constant values
    if len(group_valid) < 2 or group_valid["NINO34_SST"].nunique() < 2 or group_valid["PRECIP_ANOM"].nunique() < 2:
        group = group.copy()
        group.loc[:, "NINO34_PRECIP_ANOM"] = np.nan
        return group
    
    # Linear regression
    slope, intercept, _, _, _ = linregress(group_valid["NINO34_SST"], group_valid["PRECIP_ANOM"])
    
    # Compute predicted anomaly
    group = group.copy()
    group.loc[:, "NINO34_PRECIP_ANOM"] = slope * group["NINO34_SST"] + intercept
    return group

# Apply to each state
df_merged_state_precip_nino34 = df_merged.groupby("STATE_ABBR", group_keys=False).apply(add_nino34_pred)

/tmp/ipykernel_744755/2297600002.py:20: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_merged_state_precip_nino34 = df_merged.groupby("STATE_ABBR", group_keys=False).apply(add_nino34_pred)


# 5 Add Weather Regimes

In [16]:
# Load data
df_cluster = xr.open_dataset(cluster_file).to_dataframe().reset_index()

# Rename the index column to 'DATE' to match df_fars
df_cluster_clean = df_cluster.rename(columns={"time": "DATE"})

# Convert to datetime64
df_cluster_clean["DATE"] = pd.to_datetime(df_cluster_clean["DATE"])

# Map cluster names and drop cluster column
df_cluster_clean["CLUSTER_NAME"] = df_cluster_clean["cluster"].map(cluster_names)
df_cluster_clean=df_cluster_clean.drop(columns='cluster')

# Keep only the necessary columns from cluster dataframe
df_cluster_daily = df_cluster_clean[["DATE", "CLUSTER_NAME"]]

# Merge onto event-level dataframe
df_merged_state_precip_nino34_wxregimes = df_merged_state_precip_nino34.merge(
    df_cluster_daily,
    on="DATE",
    how="left"  # keep all events even if some dates are missing in clusters
)

# Set non-DJF regimes to NaN (i.e., months not 12, 1, 2)
df_merged_state_precip_nino34_wxregimes.loc[
    ~df_merged_state_precip_nino34_wxregimes["DATE"].dt.month.isin([12, 1, 2]),
    "CLUSTER_NAME"
] = np.nan


df_merged_state_precip_nino34_wxregimes

,DATE,STATE_ABBR,PRECIP_ANOM,FATAL_CRASH_COUNT,NINO34_SST,NINO34_PRECIP_ANOM,CLUSTER_NAME
0,1981-01-01,AL,-4.391150,0.0,-0.36,-0.058759,West Coast High
1,1981-01-02,AL,-4.417280,0.0,-0.36,-0.058759,West Coast High
2,1981-01-03,AL,-4.426943,0.0,-0.36,-0.058759,West Coast High
3,1981-01-04,AL,-4.419716,0.0,-0.36,-0.058759,West Coast High
4,1981-01-05,AL,-4.411256,0.0,-0.36,-0.058759,West Coast High
...,...,...,...,...,...,...,...
753835,2023-12-27,WY,-0.512046,0.0,2.03,0.056030,NaN
753836,2023-12-28,WY,-0.460988,0.0,2.03,0.056030,NaN
753837,2023-12-29,WY,-0.530808,0.0,2.03,0.056030,NaN
753838,2023-12-30,WY,-0.528172,0.0,2.03,0.056030,NaN


In [17]:
monthly_counts = (
    df_state_precip
    .groupby([df_state_precip['DATE'].dt.year, 
              df_state_precip['DATE'].dt.month])
    .size()
    .unstack(fill_value=0)
)
print(monthly_counts) 

DATE    1     2     3     4     5     6     7     8     9     10    11    12
DATE                                                                        
1981  1488  1344  1488  1440  1488  1440  1488  1488  1440  1488  1440  1488
1982  1488  1344  1488  1440  1488  1440  1488  1488  1440  1488  1440  1488
1983  1488  1344  1488  1440  1488  1440  1488  1488  1440  1488  1440  1488
1984  1488  1392  1488  1440  1488  1440  1488  1488  1440  1488  1440  1488
1985  1488  1344  1488  1440  1488  1440  1488  1488  1440  1488  1440  1488
1986  1488  1344  1488  1440  1488  1440  1488  1488  1440  1488  1440  1488
1987  1488  1344  1488  1440  1488  1440  1488  1488  1440  1488  1440  1488
1988  1488  1392  1488  1440  1488  1440  1488  1488  1440  1488  1440  1488
1989  1488  1344  1488  1440  1488  1440  1488  1488  1440  1488  1440  1488
1990  1488  1344  1488  1440  1488  1440  1488  1488  1440  1488  1440  1488
1991  1488  1344  1488  1440  1488  1440  1488  1488  1440  1488  1440  1488

# 6 Add Crash Anomalies and Detrended Crash Anomalies by State

In [18]:
df = df_merged_state_precip_nino34_wxregimes.copy()

# Step 1: compute daily climatology per state (average for each day-of-year)
df['DAY_OF_YEAR'] = df['DATE'].dt.dayofyear

clim = (
    df.groupby(['STATE_ABBR', 'DAY_OF_YEAR'])['FATAL_CRASH_COUNT']
      .mean()
      .rename('CLIM_CRASH')
      .reset_index()
)

df = df.merge(clim, on=['STATE_ABBR', 'DAY_OF_YEAR'], how='left')

# Step 2: compute daily anomaly
df['FATAL_CRASH_ANOM'] = df['FATAL_CRASH_COUNT'] - df['CLIM_CRASH']

# numeric time for detrending
df['DATE_ORD'] = df['DATE'].map(lambda d: d.toordinal())

# apply per state
results = (
    df.groupby('STATE_ABBR', group_keys=False)
      .apply(lambda g: pd.DataFrame(
          {'DETREND': detrend_with_trend(g['DATE_ORD'], g['FATAL_CRASH_ANOM'])[0],
           'TREND':   detrend_with_trend(g['DATE_ORD'], g['FATAL_CRASH_ANOM'])[1]},
          index=g.index
      ))
)

# assign to dataframe
df['FATAL_CRASH_ANOM_DETREND'] = results['DETREND']
df['FATAL_CRASH_ANOM_TREND'] = results['TREND']


# cleanup
df_merged_daily_state_anoms = df.drop(columns=['DAY_OF_YEAR', 'DATE_ORD'])
df_merged_daily_state_anoms["DATE"] = pd.to_datetime(df_merged_daily_state_anoms["DATE"])


# Write out daily summary
outFile='../data/combined_databases/database_daily_summary_state.csv'
print("Writing: ",outFile)
print(df_merged_daily_state_anoms.head())
df_merged_daily_state_anoms.to_csv(outFile)

/tmp/ipykernel_744755/3445207502.py:24: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.DataFrame(


Writing:  ../data/combined_databases/database_daily_summary_state.csv
        DATE STATE_ABBR  PRECIP_ANOM  FATAL_CRASH_COUNT  NINO34_SST  \
0 1981-01-01         AL    -4.391150                0.0       -0.36   
1 1981-01-02         AL    -4.417280                0.0       -0.36   
2 1981-01-03         AL    -4.426943                0.0       -0.36   
3 1981-01-04         AL    -4.419716                0.0       -0.36   
4 1981-01-05         AL    -4.411256                0.0       -0.36   

   NINO34_PRECIP_ANOM     CLUSTER_NAME  CLIM_CRASH  FATAL_CRASH_ANOM  \
0           -0.058759  West Coast High    0.604651         -0.604651   
1           -0.058759  West Coast High    0.302326         -0.302326   
2           -0.058759  West Coast High    0.162791         -0.162791   
3           -0.058759  West Coast High    0.186047         -0.186047   
4           -0.058759  West Coast High    0.511628         -0.511628   

   FATAL_CRASH_ANOM_DETREND  FATAL_CRASH_ANOM_TREND  
0               

# 7 Make National aggregated totals for daily

In [19]:
df_states = df_merged_daily_state_anoms.copy()  # df has daily state-level anomalies

# Step 1: aggregate daily totals over all states
df_national = df_states.groupby('DATE').agg({
    'FATAL_CRASH_COUNT': 'sum',
    'NINO34_SST': 'mean',
    'PRECIP_ANOM': 'mean',
    'NINO34_PRECIP_ANOM': 'mean',
    'CLUSTER_NAME': 'first'
}).reset_index()

# Step 2: compute daily climatology (average for each day-of-year)
df_national['DAY_OF_YEAR'] = df_national['DATE'].dt.dayofyear
clim_natl = (
    df_national.groupby('DAY_OF_YEAR')['FATAL_CRASH_COUNT']
      .mean()
      .rename('CLIM_CRASH')
      .reset_index()
)

# Merge climatology back
df_national = df_national.merge(clim_natl, on='DAY_OF_YEAR', how='left')

# Step 3: compute anomaly
df_national['FATAL_CRASH_ANOM'] = df_national['FATAL_CRASH_COUNT'] - df_national['CLIM_CRASH']

df_national['DATE_ORD'] = df_national['DATE'].map(lambda d: d.toordinal())

df_national['FATAL_CRASH_ANOM_DETREND'],df_national['FATAL_CRASH_ANOM_TREND'] = detrend_with_trend(df_national['DATE_ORD'], df_national['FATAL_CRASH_ANOM'])

# Cleanup
df_national_daily = df_national.drop(columns=['DAY_OF_YEAR', 'DATE_ORD'])
df_national_daily["DATE"] = pd.to_datetime(df_national_daily["DATE"])

# Write out daily national summary
outFile='../data/combined_databases/database_daily_summary_national.csv'
print("Writing: ",outFile)
print(df_national_daily.head())
df_national_daily.to_csv(outFile)

Writing:  ../data/combined_databases/database_daily_summary_national.csv
        DATE  FATAL_CRASH_COUNT  NINO34_SST  PRECIP_ANOM  NINO34_PRECIP_ANOM  \
0 1981-01-01               20.0       -0.36    -1.868683           -0.036087   
1 1981-01-02               19.0       -0.36    -1.457314           -0.036087   
2 1981-01-03               12.0       -0.36    -2.058164           -0.036087   
3 1981-01-04                3.0       -0.36    -2.195548           -0.036087   
4 1981-01-05                3.0       -0.36    -2.152618           -0.036087   

      CLUSTER_NAME  CLIM_CRASH  FATAL_CRASH_ANOM  FATAL_CRASH_ANOM_DETREND  \
0  West Coast High   20.744186         -0.744186                 -4.434514   
1  West Coast High   15.348837          3.651163                 -0.038695   
2  West Coast High   17.279070         -5.279070                 -8.968458   
3  West Coast High   14.627907        -11.627907                -15.316825   
4  West Coast High   14.046512        -11.046512        

In [20]:
df_merged_daily_state_anoms

,DATE,STATE_ABBR,PRECIP_ANOM,FATAL_CRASH_COUNT,NINO34_SST,NINO34_PRECIP_ANOM,CLUSTER_NAME,CLIM_CRASH,FATAL_CRASH_ANOM,FATAL_CRASH_ANOM_DETREND,FATAL_CRASH_ANOM_TREND
0,1981-01-01,AL,-4.391150,0.0,-0.36,-0.058759,West Coast High,0.604651,-0.604651,-0.636723,0.032072
1,1981-01-02,AL,-4.417280,0.0,-0.36,-0.058759,West Coast High,0.302326,-0.302326,-0.334393,0.032068
2,1981-01-03,AL,-4.426943,0.0,-0.36,-0.058759,West Coast High,0.162791,-0.162791,-0.194854,0.032064
3,1981-01-04,AL,-4.419716,0.0,-0.36,-0.058759,West Coast High,0.186047,-0.186047,-0.218106,0.032059
4,1981-01-05,AL,-4.411256,0.0,-0.36,-0.058759,West Coast High,0.511628,-0.511628,-0.543683,0.032055
...,...,...,...,...,...,...,...,...,...,...,...
753835,2023-12-27,WY,-0.512046,0.0,2.03,0.056030,NaN,0.046512,-0.046512,-0.052683,0.006171
753836,2023-12-28,WY,-0.460988,0.0,2.03,0.056030,NaN,0.046512,-0.046512,-0.052684,0.006172
753837,2023-12-29,WY,-0.530808,0.0,2.03,0.056030,NaN,0.000000,0.000000,-0.006173,0.006173
753838,2023-12-30,WY,-0.528172,0.0,2.03,0.056030,NaN,0.069767,-0.069767,-0.075941,0.006174


# 8 Aggregate Monthly by State

In [21]:
# Step 1: add MONTH column (first day of month)
df = df_merged_daily_state_anoms.copy()
df['MONTH'] = df['DATE'].dt.to_period('M').dt.to_timestamp()

# Step 2: define aggregation rules
agg_dict = {
    'FATAL_CRASH_COUNT': 'sum',          # total crashes in month
    'PRECIP_ANOM': 'mean',               # average precip anomaly
    'NINO34_SST': 'mean',                # average Nino3.4 SST
    'NINO34_PRECIP_ANOM': 'mean',        # average predicted precip anomaly
    'CLUSTER_NAME': lambda x: x.mode()[0] if not x.mode().empty else np.nan,
    'CLIM_CRASH': 'sum',                # mean climatology
    'FATAL_CRASH_ANOM': 'sum',          # mean anomaly
    'FATAL_CRASH_ANOM_DETREND': 'sum',   # mean detrended anomaly
    'FATAL_CRASH_ANOM_TREND': 'sum'   # mean detrended anomaly
}

# Step 3: aggregate by STATE_ABBR + MONTH
df_monthly_state = (
    df.groupby(['STATE_ABBR', 'MONTH'])
      .agg(agg_dict)
      .reset_index()
)

# reorder columns
df_monthly_state = df_monthly_state[
    ['STATE_ABBR', 'MONTH', 'FATAL_CRASH_COUNT', 'PRECIP_ANOM', 'NINO34_SST',
     'NINO34_PRECIP_ANOM', 'CLUSTER_NAME', 'CLIM_CRASH', 'FATAL_CRASH_ANOM', 'FATAL_CRASH_ANOM_TREND','FATAL_CRASH_ANOM_DETREND']
]

# Cleanup
df_monthly_state = df_monthly_state.rename(columns={"MONTH": "DATE"})
df_monthly_state["DATE"] = pd.to_datetime(df_monthly_state["DATE"])

# Write out monthly summary by state
outFile='../data/combined_databases/database_monthly_summary_state.csv'
print("Writing: ",outFile)
print(df_monthly_state.head())
df_monthly_state.to_csv(outFile)

Writing:  ../data/combined_databases/database_monthly_summary_state.csv
  STATE_ABBR       DATE  FATAL_CRASH_COUNT  PRECIP_ANOM  NINO34_SST  \
0         AL 1981-01-01                4.0    -2.909368       -0.36   
1         AL 1981-02-01                6.0     0.921696       -0.64   
2         AL 1981-03-01               12.0    -0.091817       -0.64   
3         AL 1981-04-01                2.0    -1.465434       -0.53   
4         AL 1981-05-01                5.0    -0.193220       -0.57   

   NINO34_PRECIP_ANOM    CLUSTER_NAME  CLIM_CRASH  FATAL_CRASH_ANOM  \
0           -0.058759  Greenland High   11.953488         -7.953488   
1           -0.140641  Pacific Trough   10.767442         -4.767442   
2           -0.140641             NaN    9.674419          2.325581   
3           -0.108473             NaN    7.604651         -5.604651   
4           -0.120170             NaN    7.860465         -2.860465   

   FATAL_CRASH_ANOM_TREND  FATAL_CRASH_ANOM_DETREND  
0                0.9

# 9 Aggregate Monthly Data for National Totals

In [22]:
agg_dict_national = {
    'FATAL_CRASH_COUNT': 'sum',           # total crashes nationwide
    'PRECIP_ANOM': 'mean',                # mean over all states
    'NINO34_SST': 'mean',                 # mean Nino3.4 SST
    'NINO34_PRECIP_ANOM': 'mean',         # mean predicted anomaly
    'CLUSTER_NAME': lambda x: x.mode()[0] if not x.mode().empty else np.nan,  # most frequent cluster
    'CLIM_CRASH': 'sum',                 # mean climatology
    'FATAL_CRASH_ANOM': 'sum',           # mean anomaly
    'FATAL_CRASH_ANOM_DETREND': 'sum',    # mean detrended anomaly
    'FATAL_CRASH_ANOM_TREND': 'sum'    # mean trend

}

df_monthly_national = (
    df_monthly_state
    .groupby('DATE', as_index=False)
    .agg(agg_dict_national)
)

# Optional: reorder columns
df_monthly_national = df_monthly_national[
    ['DATE', 'FATAL_CRASH_COUNT', 'PRECIP_ANOM', 'NINO34_SST',
     'NINO34_PRECIP_ANOM', 'CLUSTER_NAME', 'CLIM_CRASH', 'FATAL_CRASH_ANOM', 'FATAL_CRASH_ANOM_TREND','FATAL_CRASH_ANOM_DETREND']
]

# Cleanup
df_monthly_national["DATE"] = pd.to_datetime(df_monthly_national["DATE"])

# Write out monthly national summary 
outFile='../data/combined_databases/database_monthly_summary_national.csv'
print("Writing: ",outFile)
print(df_monthly_national.head())
df_monthly_national.to_csv(outFile)

Writing:  ../data/combined_databases/database_monthly_summary_national.csv
        DATE  FATAL_CRASH_COUNT  PRECIP_ANOM  NINO34_SST  NINO34_PRECIP_ANOM  \
0 1981-01-01              335.0    -1.356085       -0.36           -0.036087   
1 1981-02-01              474.0     0.486407       -0.64           -0.063302   
2 1981-03-01              361.0    -0.687008       -0.64           -0.063302   
3 1981-04-01              364.0    -0.296151       -0.53           -0.052610   
4 1981-05-01              389.0     0.144107       -0.57           -0.056498   

     CLUSTER_NAME  CLIM_CRASH  FATAL_CRASH_ANOM  FATAL_CRASH_ANOM_TREND  \
0  Greenland High  458.000000       -123.000000              114.181623   
1  Pacific Trough  385.581395         88.418605              102.743580   
2             NaN  368.906977         -7.906977              113.322019   
3             NaN  300.674419         63.325581              109.236433   
4             NaN  268.627907        120.372093              112.4332

In [23]:
df_monthly_national

,DATE,FATAL_CRASH_COUNT,PRECIP_ANOM,NINO34_SST,NINO34_PRECIP_ANOM,CLUSTER_NAME,CLIM_CRASH,FATAL_CRASH_ANOM,FATAL_CRASH_ANOM_TREND,FATAL_CRASH_ANOM_DETREND
0,1981-01-01,335.0,-1.356085,-0.36,-0.036087,Greenland High,458.000000,-123.000000,114.181623,-237.181623
1,1981-02-01,474.0,0.486407,-0.64,-0.063302,Pacific Trough,385.581395,88.418605,102.743580,-14.324975
2,1981-03-01,361.0,-0.687008,-0.64,-0.063302,NaN,368.906977,-7.906977,113.322019,-121.228996
3,1981-04-01,364.0,-0.296151,-0.53,-0.052610,NaN,300.674419,63.325581,109.236433,-45.910851
4,1981-05-01,389.0,0.144107,-0.57,-0.056498,NaN,268.627907,120.372093,112.433276,7.938817
...,...,...,...,...,...,...,...,...,...,...
511,2023-08-01,158.0,0.015256,1.35,0.130122,NaN,227.744186,-69.744186,-112.404137,42.659951
512,2023-09-01,183.0,-0.152229,1.60,0.154421,NaN,259.232558,-76.232558,-109.208234,32.975676
513,2023-10-01,295.0,-0.573874,1.72,0.166085,NaN,351.116279,-56.116279,-113.292880,57.176601
514,2023-11-01,246.0,-0.945780,2.02,0.195245,NaN,440.697674,-194.697674,-110.068308,-84.629367
